# **Ahmed Gaitani Code**

In [ ]:
# Import 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the data
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')

In [ ]:

# Verify the columns are as expected
print("Train columns:", train_data.columns)
print("Test columns:", test_data.columns)


In [ ]:
# Feature Engineering
def feature_engineering(df):
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
                                        'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    df['Sex'] = df['Sex'].map( {'female': 1, 'male': 0} ).astype(int)
    
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1
    
    df['Embarked'] = df['Embarked'].fillna('S')
    
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    df['Age*Class'] = df['Age'] * df['Pclass']
    
    df = df.drop(['Ticket', 'Cabin', 'Name', 'PassengerId'], axis=1)
    return df

In [ ]:
# Apply feature engineering to the train and test data
train_data = feature_engineering(train_data)
test_data = feature_engineering(test_data)

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

In [ ]:
# Split the data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Define categorical and numerical columns
categorical_features = ['Title', 'Embarked']
numerical_features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'Age*Class']

In [ ]:
# Create transformers for preprocessing
numerical_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


In [ ]:
# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# Building the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42))
])

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0],
    'classifier__colsample_bytree': [0.8, 1.0]
}

In [ ]:
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters
print("Best parameters found: ", grid_search.best_params_)

In [ ]:
# Evaluation
best_model = grid_search.best_estimator_
accuracy = cross_val_score(best_model, X_valid, y_valid, cv=5, scoring='accuracy').mean()
print("Validation Accuracy: ", accuracy)

In [ ]:
# Predicting on the test set
predictions = best_model.predict(test_data)

In [ ]:
# Creating a submission file
submission = pd.DataFrame({
    "PassengerId": pd.read_csv('/kaggle/input/titanic/test.csv')["PassengerId"],
    "Survived": predictions
})

submission.to_csv('submission.csv', index=False)

# Visualizations

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:

# Age Distribution
plt.figure(figsize=(10, 6))
sns.histplot(train_data['Age'].dropna(), bins=30, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Fare Distribution
plt.figure(figsize=(10, 6))
sns.histplot(train_data['Fare'], bins=30, kde=True)
plt.title('Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Survival Rate by Sex
plt.figure(figsize=(10, 6))
sns.barplot(x='Sex', y='Survived', data=train_data)
plt.title('Survival Rate by Sex')
plt.xlabel('Sex')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
# Survival Rate by Pclass
plt.figure(figsize=(10, 6))
sns.barplot(x='Pclass', y='Survived', data=train_data)
plt.title('Survival Rate by Pclass')
plt.xlabel('Pclass')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
# Survival Rate by Embarked
plt.figure(figsize=(10, 6))
sns.barplot(x='Embarked', y='Survived', data=train_data)
plt.title('Survival Rate by Embarked')
plt.xlabel('Embarked')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
# Pairplot
sns.pairplot(train_data[['Age', 'Fare', 'Pclass', 'Survived']], hue='Survived', diag_kind='kde')
plt.show()

In [ ]:
# Survival Rate by Family Size
plt.figure(figsize=(10, 6))
sns.barplot(x='FamilySize', y='Survived', data=train_data)
plt.title('Survival Rate by Family Size')
plt.xlabel('Family Size')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
# Survival Rate by Title
plt.figure(figsize=(12, 6))
sns.barplot(x='Title', y='Survived', data=train_data)
plt.title('Survival Rate by Title')
plt.xlabel('Title')
plt.ylabel('Survival Rate')
plt.xticks(rotation=45)
plt.show()